# Jupyter Notebook for loading the full TESS Input Catalog (TIC) from https://archive.stsci.edu/tess/tic_ctl.html and reducing it to the required information while retaining only TICIDs with magnitude $\mathrm{Tmag} \leq 10$
## This reduces the data volume from TB scale to ca. 100 MB

In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
# Define the column names of the full TIC catalog
COLUMNS = [
    "ID", "version", "HIP", "TYC", "UCAC", "TWOMASS", "SDSS", "ALLWISE",
    "GAIA", "APASS", "KIC", "objType", "typeSrc", "ra", "dec", "POSflag",
    "pmRA", "e_pmRA", "pmDEC", "e_pmDEC", "PMflag", "plx", "e_plx", "PARflag",
    "gallong", "gallat", "eclong", "eclat", "Bmag", "e_Bmag", "Vmag", "e_Vmag",
    "umag", "e_umag", "gmag", "e_gmag", "rmag", "e_rmag", "imag", "e_imag",
    "zmag", "e_zmag", "Jmag", "e_Jmag", "Hmag", "e_Hmag", "Kmag", "e_Kmag",
    "TWOMflag", "prox", "w1mag", "e_w1mag", "w2mag", "e_w2mag", "w3mag",
    "e_w3mag", "w4mag", "e_w4mag", "GAIAmag", "e_GAIAmag", "Tmag", "e_Tmag",
    "TESSflag", "SPFlag", "Teff", "e_Teff", "logg", "e_logg", "MH", "e_MH",
    "rad", "e_rad", "mass", "e_mass", "rho", "e_rho", "lumclass", "lum",
    "e_lum", "d", "e_d", "ebv", "e_ebv", "numcont", "contratio",
    "disposition", "duplicate_id", "priority", "eneg_EBV", "epos_EBV",
    "EBVflag", "eneg_Mass", "epos_Mass", "eneg_Rad", "epos_Rad",
    "eneg_rho", "epos_rho", "eneg_logg", "epos_logg", "eneg_lum",
    "epos_lum", "eneg_dist", "epos_dist", "distflag", "eneg_Teff",
    "epos_Teff", "TeffFlag", "gaiabp", "e_gaiabp", "gaiarp", "e_gaiarp",
    "gaiaqflag", "starchareFlag", "VmagFlag", "BmagFlag", "splists",
    "e_RA", "e_Dec", "RA_orig", "Dec_orig", "e_RA_orig", "e_Dec_orig",
    "raddflag", "wdflag", "objID"
]

# Define the TESS magnitude threshold for reduced TIC
Tmag_min = 10

def extract_id_ra_dec(input_csv, output_csv, chunksize=500_000):
    # Counter for the total number of retained TIC entries
    number_of_IDs = 0

    # Load only the columns required for the analysis
    usecols = ["ID", "ra", "dec", "Tmag"]

    # Define data types to reduce memory usage during catalog loading
    dtype_dict = {
        "ID": "int64",
        "ra": "float32",
        "dec": "float32",
        "Tmag": "float32",
    }

    # Track whether the current chunk is the first one written to the output file
    first_chunk = True

    # Process the large TIC catalog in chunks to avoid loading the full file into memory
    for i, chunk in enumerate(pd.read_csv(
        input_csv,
        names=COLUMNS,
        usecols=usecols,
        dtype=dtype_dict,
        chunksize=chunksize,
        low_memory=True
    )):

        # Retain only Tmag <= Tmag_min targets and remove entries with missing required values
        chunk = chunk[chunk["Tmag"] <= Tmag_min]
        chunk = chunk.dropna(subset=["Tmag", "ra", "dec", "ID"])

        # Append filtered data to the output catalog
        chunk[["ID", "ra", "dec", "Tmag"]].to_csv(
            output_csv,
            mode='w' if first_chunk else 'a',
            header=first_chunk,
            index=False
        )

        first_chunk = False
        print(f"Processed chunk {i+1} ({len(chunk)} rows after filter)")

        # Update total number of retained TIC entries
        number_of_IDs += len(chunk)

    print(f"Done. Saved to {output_csv}. {number_of_IDs} rows in total.")

In [ ]:
# Select the hemisphere and declination range to process
hemisphere = "N"
dec_num = 10

# For the southern hemisphere, select the TIC declination file covering the corresponding range
if hemisphere == "S":
    input_file = f"tic/tic_dec{dec_num}_00{hemisphere}__{dec_num-2}_00{hemisphere}.csv"

    # Extract relevant TIC information and retain only stars with Tmag <= 10
    extract_id_ra_dec(
        input_csv=input_file,
        output_csv=f"tic/TIC_reduced_{hemisphere}{dec_num}_{hemisphere}{dec_num-2}_Tmag_10.csv",
        chunksize=500_000
    )

# For the northern hemisphere, select the TIC declination file covering the corresponding range
elif hemisphere == "N":
    input_file = f"tic/tic_dec{dec_num}_00{hemisphere}__{dec_num+2}_00{hemisphere}.csv"

    # Extract relevant TIC information and retain only stars with Tmag <= 10
    extract_id_ra_dec(
        input_csv=input_file,
        output_csv=f"tic/TIC_reduced_{hemisphere}{dec_num}_{hemisphere}{dec_num+2}_Tmag_10.csv",
        chunksize=500_000
    )